In [0]:
%run ../../_shared/00_project_config

In [0]:
%run ../../_shared/01_spark_utils

In [0]:
from pyspark.sql import functions as F


# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
# Project-wide schema names and supported FBref leagues are loaded from
# 00_project_config.
#
# Reusable reading, cleaning, validation, deduplication, and writing helpers
# are loaded from 01_spark_utils.

SILVER_TABLE = f"{SILVER_SCHEMA}.team_shooting_stats"

FBREF_DATASET = "shooting_stats"

# One Silver row represents:
#
#   team + season + league
#
# for the FBref team shooting dataset.
BUSINESS_KEY = [
    "team_name",
    "season_code",
    "league_code",
]

REQUIRED_SILVER_COLUMNS = [
    "team_name",
    "season_code",
    "season_start_year",
    "league_code",
]


# -----------------------------------------------------------------------------
# Build expected Bronze table list
# -----------------------------------------------------------------------------
# FBref Bronze maintains one table per:
#
#   dataset + league
#
# Each Bronze table contains all supported seasons for that league.

BRONZE_TABLES = [
    f"{BRONZE_SCHEMA}.fbref_{FBREF_DATASET}_{league_code}"
    for league_code in FBREF_LEAGUES
]


# -----------------------------------------------------------------------------
# Read and combine Bronze tables
# -----------------------------------------------------------------------------
# read_and_union_tables() is strict by default:
#
#   - every expected league table must exist
#   - schemas must be compatible
#
# A missing league should fail the pipeline rather than silently create an
# incomplete Silver table.

bronze_df = read_and_union_tables(
    spark,
    BRONZE_TABLES,
)


# -----------------------------------------------------------------------------
# Validate expected FBref source columns
# -----------------------------------------------------------------------------
# FBref is an external HTML source and its table structure may change.
# Validate every field used by this transformation before selecting it.

REQUIRED_BRONZE_COLUMNS = [
    "Squad",
    "#_Pl",
    "90s",
    "Standard_Gls",
    "Standard_Sh",
    "Standard_SoT",
    "Standard_SoT%",
    "Standard_Sh/90",
    "Standard_SoT/90",
    "Standard_G/Sh",
    "Standard_G/SoT",
    "Standard_PK",
    "Standard_PKatt",
    "_season",
    "_league",
    "_stat",
    "_ingested_at",
    "_source_file",
    "_source_system",
]

validate_columns_exist(
    bronze_df,
    REQUIRED_BRONZE_COLUMNS,
)


# -----------------------------------------------------------------------------
# Clean top-level strings
# -----------------------------------------------------------------------------
# FBref Bronze is flat, so the shared top-level string cleaner is appropriate.
#
# It trims string values and converts empty strings to NULL.

bronze_df = clean_top_level_strings(bronze_df)


# -----------------------------------------------------------------------------
# Validate dataset identity
# -----------------------------------------------------------------------------
# All rows processed by this notebook should originate from the FBref
# "shooting_stats" dataset.

unexpected_stat_rows = (
    bronze_df
    .filter(
        F.col("_stat").isNull()
        | (F.col("_stat") != F.lit(FBREF_DATASET))
    )
    .limit(1)
    .count()
)

if unexpected_stat_rows > 0:
    raise ValueError(
        f"Unexpected _stat value found while processing {FBREF_DATASET}."
    )


# -----------------------------------------------------------------------------
# Select, rename, and cast Silver columns
# -----------------------------------------------------------------------------
# Source-oriented FBref names are converted into clear analytical names.
#
# The "90s" field is retained because it provides important context for
# per-90 shooting metrics.

silver_df = (
    bronze_df
    .select(

        # ---------------------------------------------------------------------
        # Team
        # ---------------------------------------------------------------------

        clean_string("Squad")
        .alias("team_name"),

        F.col("`#_Pl`")
        .cast("int")
        .alias("players_used_count"),

        F.col("`90s`")
        .cast("double")
        .alias("nineties_played"),


        # ---------------------------------------------------------------------
        # Shooting volume
        # ---------------------------------------------------------------------

        F.col("Standard_Gls")
        .cast("int")
        .alias("goals"),

        F.col("Standard_Sh")
        .cast("int")
        .alias("shots"),

        F.col("Standard_SoT")
        .cast("int")
        .alias("shots_on_target"),


        # ---------------------------------------------------------------------
        # Shooting efficiency
        # ---------------------------------------------------------------------

        F.col("`Standard_SoT%`")
        .cast("double")
        .alias("shots_on_target_percentage"),

        F.col("`Standard_Sh/90`")
        .cast("double")
        .alias("shots_per_90"),

        F.col("`Standard_SoT/90`")
        .cast("double")
        .alias("shots_on_target_per_90"),

        F.col("`Standard_G/Sh`")
        .cast("double")
        .alias("goals_per_shot"),

        F.col("`Standard_G/SoT`")
        .cast("double")
        .alias("goals_per_shot_on_target"),


        # ---------------------------------------------------------------------
        # Penalties
        # ---------------------------------------------------------------------

        F.col("Standard_PK")
        .cast("int")
        .alias("penalty_goals"),

        F.col("Standard_PKatt")
        .cast("int")
        .alias("penalty_kicks_attempted"),


        # ---------------------------------------------------------------------
        # Season / league context
        # ---------------------------------------------------------------------

        F.col("_season")
        .cast("string")
        .alias("season_code"),

        season_code_to_start_year("_season")
        .alias("season_start_year"),

        F.col("_league")
        .cast("string")
        .alias("league_code"),


        # ---------------------------------------------------------------------
        # Bronze lineage metadata
        # ---------------------------------------------------------------------

        F.col("_ingested_at")
        .cast("timestamp")
        .alias("_bronze_ingested_at"),

        F.col("_source_file")
        .cast("string")
        .alias("_bronze_source_file"),

        F.col("_source_system")
        .cast("string")
        .alias("_source_system"),
    )
)


# -----------------------------------------------------------------------------
# Future entity standardization placeholders
# -----------------------------------------------------------------------------
# The next project phase will standardize teams and competitions across:
#
#   API-Football
#   FBref
#   football-data.co.uk
#
# Important future candidates from this dataset:
#
#   team_name
#   league_code
#
# Preferred future model:
#
#   team_name_source
#   team_name_standardized
#   team_key
#
#   competition_key
#
# Source values should remain available for traceability. Standardization
# should add canonical identifiers rather than overwrite the original values.
#
# No physical placeholder columns are added yet because empty or duplicated
# fields would increase schema noise before the matching logic exists.


# -----------------------------------------------------------------------------
# Validate business key before deduplication
# -----------------------------------------------------------------------------
# Validate the complete business key before deduplication so rows with NULL
# team, season, or league identifiers cannot be grouped and reduced
# arbitrarily.

validate_required_not_null(
    silver_df,
    BUSINESS_KEY,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Deduplicate repeated Bronze snapshots
# -----------------------------------------------------------------------------
# Bronze currently uses append mode, so rerunning the same FBref Bronze load
# can produce repeated team/season/league records.
#
# Keep the record from the latest Bronze ingestion event.

silver_df = deduplicate_latest(
    silver_df,
    key_columns=BUSINESS_KEY,
    order_column="_bronze_ingested_at",
    tie_breaker_columns=[
        "_bronze_source_file",
    ],
)


# -----------------------------------------------------------------------------
# Add Silver processing metadata
# -----------------------------------------------------------------------------

silver_df = silver_df.withColumn(
    "_silver_processed_at",
    F.current_timestamp(),
)


# -----------------------------------------------------------------------------
# Final Silver validation
# -----------------------------------------------------------------------------
# Required analytical identifiers are validated as the final data-quality
# gate.
#
# The helper also returns the final row count, avoiding a separate count()
# action.

validation_metrics = validate_required_not_null(
    silver_df,
    REQUIRED_SILVER_COLUMNS,
    fail_on_null=True,
)


# -----------------------------------------------------------------------------
# Write Silver table
# -----------------------------------------------------------------------------
# The current project phase uses full-refresh Silver processing.

write_full_refresh(
    silver_df,
    SILVER_TABLE,
)


# -----------------------------------------------------------------------------
# Completion message
# -----------------------------------------------------------------------------

print(
    f"Successfully loaded {validation_metrics['row_count']} "
    f"team shooting-stat records into {SILVER_TABLE}."
)